# CelLab, Deliverable 3: training, validation & benchmarks on Google Colab

This notebook trains, evaluates, validates, and benchmarks all networks of the Deliverable 3 experiment.
Click **Runtime → Run all**: every model is auto-validated right after training, the full hypothesis + live-simulator + speed-benchmark suite runs automatically after Parts 3 and 4, everything syncs to Drive, and the runtime disconnects itself when done.

**Recommended runtime (Colab Pro):** *Runtime → Change runtime type → **L4 GPU***.
- **L4** is the sweet spot: native `bf16` (no `GradScaler` overhead), 12 vCPUs for fast kernel launch, and ~5× cheaper in compute units than an A100 (which sits idle on <1M-parameter models). **T4** also works automatically (`fp16` fallback).
- Training writes to fast local SSD (`/content/results`) and syncs each finished model to Google Drive so Drive FUSE never bottlenecks epochs.
- Running the validation & GPU speed benchmark automatically on Colab takes only **~10–15 s** (~0.008 compute units on L4). You can also re-run `python -m ml.verify results/` locally for free after downloading from Drive to get laptop CPU inference timings.

| Part | What it does | Rough time on L4 |
|---|---|---|
| 1 | Set-up: code, data, Drive sync | 30 s |
| 2 | One-step network on all 256 elementary rules (+ live engine validation) | 15 s |
| 3 | Five-questions network: shared + 5 separate (+ intermediate verification) | 2–4 min |
| 4 | The 15 2D automata + 1D traffic (step + answers, fault-tolerant) | 3–5 min |
| 5 | Summary tables and horizon plots | 5 s |
| 6 | Full post-training verification & 1D/2D speed benchmarks | 10–15 s |

## 1. Set-up

In [ ]:
REPO = "https://github.com/driedpampas/cellab.git"
BRANCH = "main"
SAVE_TO_DRIVE = True          # False: keep results only in this session
AUTO_BENCHMARK = True         # True: auto-validate & benchmark on GPU (~10-15s, <0.01 credits)
AUTO_DISCONNECT = True        # True: unassign Colab runtime right after final Drive sync

import os, shutil, subprocess, sys, time
t_session_start = time.time()

if not os.path.isdir("/content/cellab"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/cellab"], check=True)
else:
    subprocess.run(["git", "-C", "/content/cellab", "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir("/content/cellab")
for _p in ("/content/cellab", "/content/cellab/src"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import importlib
sys.modules.pop("cellab", None)
importlib.invalidate_caches()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "-e", "."], check=True)

LOCAL_RESULTS = "/content/results"
DRIVE_RESULTS = "/content/drive/MyDrive/cellab_results"
os.makedirs(LOCAL_RESULTS, exist_ok=True)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    # Restore any previously finished runs from Drive so re-runs skip them cleanly.
    restored = 0
    for entry in os.listdir(DRIVE_RESULTS):
        src = os.path.join(DRIVE_RESULTS, entry)
        dst = os.path.join(LOCAL_RESULTS, entry)
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            restored += 1
    if restored:
        print(f"Restored {restored} existing result folders from Drive")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    bf16 = torch.cuda.is_bf16_supported()
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB VRAM) | bf16: {bf16} | CPUs: {os.cpu_count()}")
else:
    print("GPU: NONE - switch to L4 or T4 GPU runtime before training!")
print("Local scratch:", LOCAL_RESULTS, "| Drive backup:", DRIVE_RESULTS if SAVE_TO_DRIVE else "disabled")

The datasets are in `data/` in the repository. If one is missing (or you changed a file in `datasets/`),
this cell generates it again; the result is identical because every sample has its own seed.

In [ ]:
import glob
workers = max(1, min(os.cpu_count() or 2, 8))
for spec in sorted(glob.glob("datasets/*.json")):
    name = os.path.basename(spec)[:-5]
    if not os.path.exists(f"data/{name}.npz"):
        print("generating", name)
        subprocess.run(["cellab", "dataset", spec, "--workers", str(workers)], check=True)
print(len(glob.glob("data/*.npz")), "datasets ready")

In [ ]:
from pathlib import Path
from ml.data import load as load_data
from ml.evaluate import evaluate_and_save
from ml.train import TASKS, train
from ml.verify import print_report, run_verification, validate_model

_DATA_CACHE = {}
RUN_ERRORS = []

def get_data(dataset: str, cache: bool = True):
    if cache and dataset in _DATA_CACHE:
        return _DATA_CACHE[dataset]
    data = load_data(f"data/{dataset}.npz", DEVICE)
    if cache:
        _DATA_CACHE[dataset] = data
    return data

def sync_dir_to_drive(name: str):
    if not SAVE_TO_DRIVE:
        return
    src = Path(LOCAL_RESULTS) / name
    dst = Path(DRIVE_RESULTS) / name
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        if item.name == "last.pt" and (src / "metrics.json").exists():
            continue
        shutil.copy2(item, dst / item.name)

def sync_file_to_drive(filename: str):
    if not SAVE_TO_DRIVE:
        return
    src = Path(LOCAL_RESULTS) / filename
    if src.exists():
        shutil.copy2(src, Path(DRIVE_RESULTS) / filename)

def sync_verification_artifacts():
    for fname in ("summary.csv", "verification.json", "verification_report.md",
                  "multitask_vs_single.png", "speed_benchmark.png"):
        sync_file_to_drive(fname)

def train_and_evaluate(kind, dataset, name, epochs, batch=512, tasks=TASKS, also=(),
                       save_every=10, log_every=20, cache=True):
    out = Path(LOCAL_RESULTS) / name
    expected = [out / "metrics.json", *(Path(LOCAL_RESULTS) / f"{name}_on_{o}" / "metrics.json" for o in also)]
    if all(p.exists() for p in expected):
        print(f"[skip] {name} already complete")
        if AUTO_BENCHMARK:
            validate_model(LOCAL_RESULTS, name, datasets_dir="datasets", device=DEVICE)
        return

    t0 = time.time()
    print(f"\n=== {name} ({kind}, {dataset}, epochs={epochs}, batch={batch}, tasks={list(tasks) if kind=='answers' else ['step']}) ===")
    data = get_data(dataset, cache=cache)
    train(kind, data, str(out), epochs=epochs, batch=batch, tasks=tuple(tasks),
          device=DEVICE, resume=True, save_every=save_every, log_every=log_every)
    evaluate_and_save(kind, data, out / "best.pt", out, device=DEVICE)
    sync_dir_to_drive(name)

    for other in also:
        other_out = Path(LOCAL_RESULTS) / f"{name}_on_{other}"
        evaluate_and_save(kind, get_data(other, cache=cache), out / "best.pt", other_out, device=DEVICE)
        sync_dir_to_drive(f"{name}_on_{other}")

    if AUTO_BENCHMARK:
        v = validate_model(LOCAL_RESULTS, name, datasets_dir="datasets", device=DEVICE)
        if v["status"] == "FAIL":
            RUN_ERRORS.append(f"validation FAILED for {name}: {v['audit']['notes']}")

    print(f"=== {name} finished in {time.time() - t0:.1f}s ===")

## 2. The one-step network

It learns a single step, `row + rule → next row`, for all 256 rules at once, and answers "after N steps"
by applying itself N times. 15% of the rule families (Rule 30's among them) are never shown in training.
The same trained network is then tested on rows of 32 cells, twice as wide as anything it was trained on.

In [ ]:
train_and_evaluate("step", "elementary_w16", "step_elementary", epochs=40, also=["elementary_w32_test"], log_every=10)

## 3. The five-questions network

One pass, no simulation: the row, the rule, N and a configuration C go in, and five heads answer.
First one shared network, then five networks that each learn a single question (the comparison
promised in Deliverable 1: one network or separate ones?).

In [ ]:
train_and_evaluate("answers", "elementary_w16", "answers_elementary", epochs=80, also=["elementary_w32_test"])

In [ ]:
for task in ("state", "count", "stable", "cycle", "occurs"):
    train_and_evaluate("answers", "elementary_w16", f"only_{task}_elementary", epochs=80, tasks=(task,))

# Checkpoint verification right after the main elementary experiment finishes
if AUTO_BENCHMARK:
    print("\n--- Automatic post-Part-3 verification (elementary rules) ---")
    elem_verif = run_verification(LOCAL_RESULTS, datasets_dir="datasets", device=DEVICE, save=True)
    print_report(elem_verif)
    sync_verification_artifacts()

## 4. The 2D automata (and 1D traffic)

One model per automaton. The one-step network is trained where a step is fully determined by the grid
(deterministic automata without layers); the five-questions network is trained for all of them.
Because these datasets have 800–2,400 training samples (vs. 34,880 for elementary rules), we use
`batch=128` and `60`/`80` epochs so every model gets ~600–1,500 gradient updates instead of 80–200.
Each model runs inside a fault-tolerant guard so an error in one dataset never prevents the final
summary, verification benchmarks, Drive sync, or auto-disconnect.

In [ ]:
import json, traceback

# Release cached elementary tensors before training 2D models
_DATA_CACHE.clear()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TWO_D = sorted(os.path.basename(p)[:-5] for p in glob.glob("data/*_16.json") + glob.glob("data/traffic_w16.json"))
for name in TWO_D:
    info = json.load(open(f"data/{name}.json"))
    if info["arrays"]["states"]["type"].startswith("float"):
        print("skipping", name, "(decimal values)")
        continue
    if info["automaton"]["deterministic"] and not info["automaton"]["layers"]:
        try:
            train_and_evaluate("step", name, f"step_{name}", epochs=60, batch=128, cache=False)
        except Exception as exc:
            RUN_ERRORS.append(f"step_{name}: {exc}")
            traceback.print_exc()
    try:
        train_and_evaluate("answers", name, f"answers_{name}", epochs=80, batch=128, cache=False)
    except Exception as exc:
        RUN_ERRORS.append(f"answers_{name}: {exc}")
        traceback.print_exc()

## 5. Summary tables and plots

In [ ]:
import pandas as pd
rows = []
for path in sorted(glob.glob(f"{LOCAL_RESULTS}/*/metrics.json")):
    report = json.load(open(path))
    for split, groups in report["splits"].items():
        for group, scores in groups.items():
            if "by_horizon" not in scores:
                continue
            row = {"model": os.path.basename(os.path.dirname(path)), "split": split, "group": group}
            for h in scores["by_horizon"]:
                row[f"cells N={h['n']}"] = round(h["model"]["cells_correct"], 3)
            last = scores["by_horizon"][-1]
            row["grids (largest N)"] = round(last["model"]["grids_correct"], 3)
            row["copy baseline (largest N)"] = round(last["copy"]["cells_correct"], 3)
            row["count error (largest N)"] = round(last["model"]["count_error"], 2)
            for key in ("stable", "occurs"):
                if key in scores:
                    row[f"{key} accuracy"], row[f"{key} F1"] = round(scores[key]["accuracy"], 3), round(scores[key]["f1"], 3)
            for key in ("cycle_start", "cycle_length"):
                if key in scores:
                    row[f"{key} error"], row[f"{key} common error"] = round(scores[key]["error"], 2), round(scores[key]["common_error"], 2)
            rows.append(row)
table = pd.DataFrame(rows)
table.to_csv(f"{LOCAL_RESULTS}/summary.csv", index=False)
sync_file_to_drive("summary.csv")
pd.set_option("display.max_rows", 500, "display.width", 250)
table[table.group == "all"]

In [ ]:
# The main result: the elementary rules by Wolfram class, for both kinds of network.
table[table.model.isin(["step_elementary", "answers_elementary"])].sort_values(["split", "group", "model"])

In [ ]:
from IPython.display import Image, display
for name in ("step_elementary", "answers_elementary", "step_elementary_on_elementary_w32_test"):
    picture = f"{LOCAL_RESULTS}/{name}/accuracy_by_horizon.png"
    if os.path.exists(picture):
        print(name); display(Image(picture))

## 6. Automatic post-training verification & benchmarks

Runs automatically after training (~10–15 s on L4) and saves `verification.json`, `verification_report.md`,
`multitask_vs_single.png`, and `speed_benchmark.png` to Drive:
1. **Integrity & convergence audit** across every trained model.
2. **Core Deliverable 3 hypotheses** (StepNet 100% seen/unseen/32-cell exactness; AnswerNet Wolfram-class ordering & chaotic collapse; shared vs. 5 separate networks).
3. **Live simulator cross-check** (`StepNet` checkpoints vs. `cellab.engine.step` on fresh grids, including unseen Rule 30 on width 16 & 32 and deterministic 2D automata).
4. **Inference speed benchmarks** (CPU simulator vs. GPU `StepNet` rollout vs. GPU `AnswerNet` single-pass, for both 1D elementary and 2D 16×16).

In [ ]:
if AUTO_BENCHMARK:
    t_verif = time.time()
    verification = run_verification(LOCAL_RESULTS, datasets_dir="datasets", device=DEVICE, save=True)
    print_report(verification)
    print(f"\nVerification & benchmarks completed in {time.time() - t_verif:.1f}s")
    sync_verification_artifacts()

    if verification["multitask_vs_single"]:
        display(pd.DataFrame(verification["multitask_vs_single"]))
    if verification["speed_benchmarks"]:
        display(pd.DataFrame(verification["speed_benchmarks"]))

    for plot_name in ("multitask_vs_single.png", "speed_benchmark.png"):
        p = f"{LOCAL_RESULTS}/{plot_name}"
        if os.path.exists(p):
            display(Image(p))
else:
    print("AUTO_BENCHMARK is False — run `python -m ml.verify results/` locally after downloading from Drive.")

if RUN_ERRORS:
    print("\nWARNINGS / ERRORS recorded during session:")
    for err in RUN_ERRORS:
        print(" -", err)

Everything is now saved locally and synced to Drive (`cellab_results/`): per-model `best.pt`, `history.json`,
`metrics.json`, `accuracy_by_horizon.png`, plus `summary.csv`, `verification.json`, `verification_report.md`,
`multitask_vs_single.png`, and `speed_benchmark.png`.

Download `cellab_results/` into `results/` in the repository. You can also run `python -m ml.verify results/`
locally for free afterwards to benchmark laptop CPU inference speed next to the Colab GPU timings.

In [ ]:
print(f"Total notebook runtime: {(time.time() - t_session_start) / 60:.1f} min")
if AUTO_DISCONNECT:
    print("All results synced to Drive. Unassigning Colab runtime in 5s to save compute units...")
    time.sleep(5)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print("Could not auto-disconnect:", exc)